# Proof Step Graph Visualization

Render proof step graphs as static matplotlib plots or interactive pyvis HTML (inline).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import HTML, display
from pyvis.network import Network

sys.path.insert(0, str(Path.cwd().parent))
from proof_step_graph.graph import (
    ProofStepGraph, NODE_GOAL, NODE_TACTIC, EDGE_INPUT, EDGE_OUTPUT,
)

# ── Colours ──────────────────────────────────────────────────────────────────
GOAL_COLOR        = "#4C9BE8"
TACTIC_COLOR      = "#F4A261"
INITIAL_COLOR     = "#2A9D8F"
EDGE_INPUT_COLOR  = "#888888"
EDGE_OUTPUT_COLOR = "#E76F51"

def _truncate(s: str, n: int) -> str:
    s = s.replace("\n", " ")
    return s if len(s) <= n else s[:n - 1] + "..."

In [ ]:
# ── Load graphs ──────────────────────────────────────────────────────────────
JSONL_PATH = Path("../data/YOUR_GRAPHS.jsonl")  # <-- change this

graphs = []
with JSONL_PATH.open() as f:
    for line in f:
        line = line.strip()
        if line:
            graphs.append(ProofStepGraph.from_dict(json.loads(line)))

print(f"Loaded {len(graphs)} graphs")
# Optionally filter by name:
# graphs = [pg for pg in graphs if pg.theorem_name in {"Nat.add_comm", "List.reverse_nil"}]

## Static matplotlib plot

In [ ]:
def _dot_layout(G: nx.DiGraph) -> dict:
    try:
        pos = nx.drawing.nx_pydot.pydot_layout(G, prog="dot")
        if pos:
            return pos
    except Exception:
        pass
    # Fallback: BFS hierarchical
    level, queue = {}, [n for n in G.nodes() if G.in_degree(n) == 0]
    for r in queue:
        level[r] = 0
    while queue:
        nxt = []
        for n in queue:
            for c in G.successors(n):
                if c not in level:
                    level[c] = level[n] + 1
                    nxt.append(c)
        queue = nxt
    mx = max(level.values(), default=-1) + 1
    for n in G.nodes():
        if n not in level:
            level[n] = mx
    by_level: dict[int, list] = {}
    for n, lv in level.items():
        by_level.setdefault(lv, []).append(n)
    return {
        n: ((i - (len(ns) - 1) / 2.0) * 2.0, -float(lv) * 1.5)
        for lv, ns in by_level.items() for i, n in enumerate(ns)
    }


def plot_static(pg: ProofStepGraph, max_label: int = 40):
    G = pg.G
    if G.number_of_nodes() == 0:
        return

    pos = _dot_layout(G)
    goal_nodes   = [n for n, d in G.nodes(data=True) if d.get("type") == NODE_GOAL and not d.get("is_initial")]
    init_nodes   = [n for n, d in G.nodes(data=True) if d.get("type") == NODE_GOAL and d.get("is_initial")]
    tactic_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == NODE_TACTIC]
    input_edges  = [(u, v) for u, v, d in G.edges(data=True) if d.get("type") == EDGE_INPUT]
    output_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get("type") == EDGE_OUTPUT]

    n = G.number_of_nodes()
    xs = [x for x, y in pos.values()]
    ys = [y for x, y in pos.values()]
    fig_w = max(10, min((max(xs) - min(xs) + 1) / 60, 40))
    fig_h = max(6,  min((max(ys) - min(ys) + 1) / 60, 60))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.set_facecolor("#1a1a2e")
    fig.patch.set_facecolor("#1a1a2e")
    ax.axis("off")

    node_size = max(300, 1200 - n * 6)
    nx.draw_networkx_nodes(G, pos, nodelist=init_nodes,   node_color=INITIAL_COLOR, node_shape="o", ax=ax, node_size=int(node_size * 1.2))
    nx.draw_networkx_nodes(G, pos, nodelist=goal_nodes,   node_color=GOAL_COLOR,    node_shape="o", ax=ax, node_size=node_size)
    nx.draw_networkx_nodes(G, pos, nodelist=tactic_nodes, node_color=TACTIC_COLOR,  node_shape="s", ax=ax, node_size=int(node_size * 0.85))

    nx.draw_networkx_edges(G, pos, edgelist=input_edges,  edge_color=EDGE_INPUT_COLOR,
                           ax=ax, arrows=True, arrowsize=12, width=1.2, connectionstyle="arc3,rad=0.05")
    nx.draw_networkx_edges(G, pos, edgelist=output_edges, edge_color=EDGE_OUTPUT_COLOR,
                           ax=ax, arrows=True, arrowsize=12, width=1.2, connectionstyle="arc3,rad=0.05")

    font_size = max(5, 9 - int(n / 20))
    label_len = max(15, max_label - int(n / 5))
    goal_labels   = {n: _truncate(G.nodes[n].get("target", n), label_len) for n in goal_nodes + init_nodes}
    tactic_labels = {n: _truncate(G.nodes[n].get("tactic", n), label_len) for n in tactic_nodes}
    nx.draw_networkx_labels(G, pos, labels=goal_labels,   font_size=font_size, font_color="white", ax=ax)
    nx.draw_networkx_labels(G, pos, labels=tactic_labels, font_size=font_size, font_color="#1a1a2e", ax=ax)

    legend_items = [
        mpatches.Patch(color=INITIAL_COLOR, label="initial goal"),
        mpatches.Patch(color=GOAL_COLOR,    label="goal"),
        mpatches.Patch(color=TACTIC_COLOR,  label="tactic"),
        mpatches.Patch(color=EDGE_INPUT_COLOR,  label="goal->tactic"),
        mpatches.Patch(color=EDGE_OUTPUT_COLOR, label="tactic->goal"),
    ]
    ax.legend(handles=legend_items, loc="upper left", fontsize=8,
              facecolor="#2a2a4e", labelcolor="white", framealpha=0.8)
    ax.set_title(pg.theorem_name, color="white", fontsize=13, pad=12)
    fig.tight_layout()
    plt.show()

In [ ]:
# Plot first few graphs (change range as needed)
for pg in graphs[:3]:
    s = pg.stats()
    print(f"{pg.theorem_name}: {s['n_goals']} goals, {s['n_tactics']} tactics, branch={s['max_branching']}")
    plot_static(pg)

## Interactive pyvis (inline HTML)

In [ ]:
def show_interactive(pg: ProofStepGraph, max_label: int = 40):
    net = Network(
        height="600px", width="100%",
        directed=True, bgcolor="#1a1a2e", font_color="white",
        notebook=True, cdn_resources="in_line",
    )
    net.set_options("""
    {
      "layout": {
        "hierarchical": {
          "enabled": true, "direction": "LR",
          "sortMethod": "directed", "nodeSpacing": 130, "levelSeparation": 200
        }
      },
      "physics": { "enabled": false },
      "edges": { "arrows": { "to": { "enabled": true, "scaleFactor": 0.8 } }, "smooth": { "type": "cubicBezier" } },
      "nodes": { "font": { "size": 13 }, "borderWidth": 2 },
      "interaction": { "navigationButtons": true, "keyboard": true, "hover": true }
    }
    """)

    for node_id, data in pg.G.nodes(data=True):
        ntype = data.get("type")
        if ntype == NODE_GOAL:
            target = _truncate(data.get("target", ""), max_label)
            case = data.get("case_name")
            label = f"case {case}\n{target}" if case else target
            color = INITIAL_COLOR if data.get("is_initial") else GOAL_COLOR
            shape, size = "ellipse", 20
            title = (f"<b>Goal</b><br><code>{data.get('target', '')}</code><br>"
                     + (f"case: {case}<br>" if case else "")
                     + (f"vars: {'; '.join(data.get('variables', []))}<br>" if data.get('variables') else ""))
        else:
            label = _truncate(data.get("tactic", ""), max_label)
            color, shape, size = TACTIC_COLOR, "box", 15
            used = data.get("used_constants", [])
            title = (f"<b>Tactic [{data.get('step_idx', '?')}]</b><br><code>{data.get('tactic', '')}</code><br>"
                     + (f"uses: {', '.join(used[:5])}" if used else ""))
        net.add_node(node_id, label=label, color=color, shape=shape, title=title, size=size)

    for src, dst, data in pg.G.edges(data=True):
        color = EDGE_OUTPUT_COLOR if data.get("type") == EDGE_OUTPUT else EDGE_INPUT_COLOR
        net.add_edge(src, dst, color=color, width=2)

    html = net.generate_html()
    display(HTML(f"<h4>{pg.theorem_name}</h4>" + html))

In [ ]:
# Show first few graphs interactively (change range as needed)
for pg in graphs[:3]:
    show_interactive(pg)